# Team MLB · SVM 확률 파일 저장 (실험 6용)

실험 6(soft voting)에 SVM을 넣으려고, **실험 4 SVM 설정 그대로** `probability=True`로 다시 학습해 확률을 저장한다.

- 입력·전처리·C·gamma는 `MLB_SVM_exp345.ipynb` 실험 4와 같다. (특징 110개, g1~g5 `log1p` → 상수 특징 제거 → 표준화, RBF)
  - Closed (log2C 7, log2γ −6) / 이진 (3, −4) / 다중 (7, −6)
- fold는 모델팀 공통 파일 `results/cv_folds_seed42.csv`를 쓴다.
- 저장: `results/svm/svm_oof_{과제}.csv`(record_id, y, fold, p_*), `svm_test_proba_{과제}.csv`(record_id, y, p_*)
- 예상 시간(Colab CPU): Closed 수 분, 이진 수 분, 다중 10~20분. 파일이 이미 있으면 건너뛴다.

`probability=True`는 내부에서 점수를 확률로 바꾸는 단계(Platt scaling)를 추가로 학습한다. 확률의 최댓값으로 고른 예측은 실험 4의 `predict`와 아주 조금 다를 수 있다. 실험 4의 공식 수치는 그대로 두고, 이 파일은 앙상블에만 쓴다.

## 0. 설정

In [1]:
FEATURE_DIR  = "/content/drive/MyDrive/MLB-Project/feature"
RESULTS_ROOT = "/content/drive/MyDrive/MLB-Project/results"
RESULT_DIR   = f"{RESULTS_ROOT}/svm"
SPLIT_PATH   = f"{RESULTS_ROOT}/split_seed42.csv"
FOLD_PATH    = f"{RESULTS_ROOT}/cv_folds_seed42.csv"
UNMON_NAME   = "unmon10k"
TRIM, SEED, N_JOBS = 0, 42, -1

BEST = {"closed": (7, -6), "binary": (3, -4), "multi": (7, -6)}   # 실험 4에서 고른 (log2C, log2γ)
SKIP_IF_EXISTS = True

In [2]:
import os, sys, time, warnings
warnings.filterwarnings("ignore", category=FutureWarning)
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
import numpy as np, pandas as pd
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.feature_selection import VarianceThreshold
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import accuracy_score, roc_auc_score
os.makedirs(RESULT_DIR, exist_ok=True)

Mounted at /content/drive


## 1. 데이터 · 분할 · fold

In [3]:
INFO_COLS = ["record_id", "dataset", "page_id", "sample_idx", "label", "trim"]
def load(kind):
    df = pd.read_csv(os.path.join(FEATURE_DIR, f"features_{kind}_trim{TRIM}.csv"))
    assert list(df.columns[:6]) == INFO_COLS
    return df
mon, unmon = load("mon"), load(UNMON_NAME)
FEATS = [c for c in mon.columns if c[:3] in ("g1_", "g2_", "g3_", "g4_", "g5_", "g6_")]
assert len(FEATS) == 110
LOG_COLS = [c for c in FEATS if not c.startswith("g6_")]

split = pd.read_csv(SPLIT_PATH); SPLIT = dict(zip(split.record_id, split.split))
for d in (mon, unmon):
    d["split"] = d.record_id.map(SPLIT); assert d.split.notna().all()
folds = pd.read_csv(FOLD_PATH)
print(len(FEATS), "features |", folds.groupby("task").size().to_dict())

110 features | {'binary': 23200, 'closed': 15200, 'multi': 23200}


## 2. 함수 (파이프라인은 실험 4와 동일)

In [4]:
def safe_log1p(X):
    return np.log1p(np.clip(X, 0, None))

def make_svm_prob(C, gamma):
    pre = ColumnTransformer([("log", FunctionTransformer(safe_log1p, feature_names_out="one-to-one"), LOG_COLS)],
                            remainder="passthrough", verbose_feature_names_out=False)
    return Pipeline([("pre", pre), ("var", VarianceThreshold()), ("sc", StandardScaler()),
                     ("svc", SVC(kernel="rbf", C=C, gamma=gamma, probability=True, random_state=SEED, cache_size=1000))])

def task_frames(task):
    if task == "closed":
        d = mon.copy(); d["y"] = d.label
    elif task == "binary":
        d = pd.concat([mon, unmon], ignore_index=True); d["y"] = np.where(d.dataset == "mon", 1, -1)
    else:
        d = pd.concat([mon, unmon], ignore_index=True); d["y"] = d.label
    d = d.sort_values("record_id").reset_index(drop=True)
    return d[d.split == "train"].reset_index(drop=True), d[d.split == "test"].reset_index(drop=True)

def save_svm_probs(task):
    p_oof = os.path.join(RESULT_DIR, f"svm_oof_{task}.csv")
    p_te  = os.path.join(RESULT_DIR, f"svm_test_proba_{task}.csv")
    if SKIP_IF_EXISTS and os.path.exists(p_oof) and os.path.exists(p_te):
        print(f"[{task}] 파일이 이미 있어 건너뜀"); return
    t0 = time.time()
    tr, te = task_frames(task)
    fold = folds[folds.task == task].set_index("record_id").fold.loc[tr.record_id].to_numpy()
    splits = [(np.where(fold != k)[0], np.where(fold == k)[0]) for k in range(5)]
    lc, lg = BEST[task]
    oof = cross_val_predict(make_svm_prob(2.0 ** lc, 2.0 ** lg), tr[FEATS], tr.y, cv=splits,
                            method="predict_proba", n_jobs=N_JOBS)
    model = make_svm_prob(2.0 ** lc, 2.0 ** lg).fit(tr[FEATS], tr.y)
    te_p = model.predict_proba(te[FEATS])
    classes = model.classes_
    cols = [f"p_{c}" for c in classes]
    pd.concat([tr[["record_id", "y"]].assign(fold=fold), pd.DataFrame(oof, columns=cols)], axis=1).to_csv(p_oof, index=False)
    pd.concat([te[["record_id", "y"]], pd.DataFrame(te_p, columns=cols)], axis=1).to_csv(p_te, index=False)
    acc_o = accuracy_score(tr.y, classes[oof.argmax(1)]); acc_t = accuracy_score(te.y, classes[te_p.argmax(1)])
    msg = f"[{task}] OOF acc {acc_o:.4f} | test acc {acc_t:.4f}"
    if task == "binary":
        i1 = list(classes).index(1)
        msg += f" | test ROC-AUC {roc_auc_score(te.y == 1, te_p[:, i1]):.4f}"
    print(msg + f" | {time.time() - t0:.0f}s 저장 완료")

## 3. Closed-world · 이진

In [5]:
save_svm_probs("closed")
save_svm_probs("binary")

[closed] OOF acc 0.8438 | test acc 0.8503 | 302s 저장 완료


/usr/local/lib/python3.13/dist-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[binary] OOF acc 0.8907 | test acc 0.8983 | test ROC-AUC 0.9524 | 1102s 저장 완료


## 4. 다중 (가장 오래 걸림)

In [6]:
save_svm_probs("multi")
print(sorted(f for f in os.listdir(RESULT_DIR) if f.startswith(("svm_oof_", "svm_test_proba_"))))

[multi] OOF acc 0.8205 | test acc 0.8328 | 995s 저장 완료
['svm_oof_binary.csv', 'svm_oof_closed.csv', 'svm_oof_multi.csv', 'svm_test_proba_binary.csv', 'svm_test_proba_closed.csv', 'svm_test_proba_multi.csv']


## 다음

`MLB_Experiment6_Ensemble.ipynb`를 위에서부터 다시 실행하면 SVM이 자동으로 들어가서 LR+SVM, SVM+RF, LR+SVM+RF 조합까지 계산된다.

참고로 test 정확도는 실험 4(Closed 0.8521 / 이진 0.8984 / 다중 0.8384)와 소수 셋째 자리 정도 다를 수 있다(확률 최댓값으로 예측하기 때문).